In [146]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import sqlite3
from scipy import stats
from itertools import combinations
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from xgboost import XGBRegressor, XGBClassifier
from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import r2_score, mean_absolute_error,mean_squared_error
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

Case Study:

John is the Head of Product at an Alberta, Canada telecommunications company.

**He wants to know the following information:**

Which customers are most likely to leave the company and why

How many people are we losing each year (on average)

What are your recommendations (if any)

In [147]:
alca_df = pd.read_csv("raw_cust_data.csv")

## Exploring Data

In [148]:
alca_df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0.0,Yes,No,1.0,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,NaN,0.0,No,No,34.0,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0.0,No,No,2.0,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0.0,No,No,45.0,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,NaN,42.30,1840.75,No
4,NaN,0.0,No,No,2.0,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [149]:
alca_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            3463 non-null   object 
 1   SeniorCitizen     7040 non-null   float64
 2   Partner           6692 non-null   object 
 3   Dependents        6684 non-null   object 
 4   tenure            6359 non-null   float64
 5   PhoneService      7041 non-null   object 
 6   MultipleLines     7039 non-null   object 
 7   InternetService   7037 non-null   object 
 8   OnlineSecurity    7041 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7040 non-null   object 
 11  TechSupport       7042 non-null   object 
 12  StreamingTV       7042 non-null   object 
 13  StreamingMovies   7040 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7041 non-null   object 
 16  PaymentMethod     3565 non-null   object 


In [150]:
alca_df.isnull().sum()

gender              3580
SeniorCitizen          3
Partner              351
Dependents           359
tenure               684
PhoneService           2
MultipleLines          4
InternetService        6
OnlineSecurity         2
OnlineBackup           0
DeviceProtection       3
TechSupport            1
StreamingTV            1
StreamingMovies        3
Contract               0
PaperlessBilling       2
PaymentMethod       3478
MonthlyCharges         0
TotalCharges           2
Churn                  2
dtype: int64

In [151]:
alca_df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7040.000000,6359.000000,7043.000000
mean,0.162216,32.366567,64.761692
std,0.368675,24.586235,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [152]:
alca_df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0.0,Yes,No,1.0,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,NaN,0.0,No,No,34.0,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0.0,No,No,2.0,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0.0,No,No,45.0,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,NaN,42.30,1840.75,No
4,NaN,0.0,No,No,2.0,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [153]:
alca_df['TotalCharges'] = pd.to_numeric(alca_df['TotalCharges'], errors='coerce')

In [154]:
alca_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            3463 non-null   object 
 1   SeniorCitizen     7040 non-null   float64
 2   Partner           6692 non-null   object 
 3   Dependents        6684 non-null   object 
 4   tenure            6359 non-null   float64
 5   PhoneService      7041 non-null   object 
 6   MultipleLines     7039 non-null   object 
 7   InternetService   7037 non-null   object 
 8   OnlineSecurity    7041 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7040 non-null   object 
 11  TechSupport       7042 non-null   object 
 12  StreamingTV       7042 non-null   object 
 13  StreamingMovies   7040 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7041 non-null   object 
 16  PaymentMethod     3565 non-null   object 


In [155]:
alca_df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0.0,Yes,No,1.0,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,NaN,0.0,No,No,34.0,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0.0,No,No,2.0,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0.0,No,No,45.0,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,NaN,42.30,1840.75,No
4,NaN,0.0,No,No,2.0,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [156]:
# from sklearn.preprocessing import LabelEncoder

# encoder = OneHotEncoder()

# cols_to_encode = ['Partner', 'Dependents', 'PhoneService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'Churn']
# encoders = {}   # store to inverse_transform later

# for col in cols_to_encode:
#     le = OneHotEncoder()
#     alca_df[[col]] = le.fit_transform(alca_df[[col]])
#     encoders[col] = le


In [157]:
# cols_to_encode = ['Partner', 'Dependents', 'PhoneService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'Churn']

# ct = ColumnTransformer(transformers=[
#     ('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cols_to_encode)
# ], remainder='passthrough')   # keeps 'price' untouched

# encoded = ct.fit_transform(alca_df)
# encoded

In [158]:
# ohe_cols = ct.named_transformers_['ohe'].get_feature_names_out(cols_to_encode)
# ohe_cols


In [159]:
# df_enc = pd.DataFrame(encoded, columns=ohe_cols)

In [160]:
# alca_df_cols = ['Partner', 'Dependents', 'PhoneService', 'OnlineSecurity',
#     'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
#     'StreamingMovies', 'PaperlessBilling', 'Churn']
# for col in alca_df_cols:
#     alca_df[col] = alca_df[col].astype(str).str.strip().str.lower().map({"no": 0, "yes": 1, "No internet service": 0})

In [161]:
alca_df_cols = ['Partner', 'Dependents', 'PhoneService', 'OnlineSecurity',
    'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
    'StreamingMovies', 'PaperlessBilling', 'Churn']
for col in alca_df_cols:
    alca_df[col] = alca_df[col].astype(str).str.strip().str.lower().map({"no": 0, "yes": 1, "no internet service": 0})
alca_df['MultipleLines'] = alca_df['MultipleLines'].astype(str).str.strip().str.lower().map({"no": 0, "yes": 1, "no phone service": 0})

In [162]:
alca_df.sample(10)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
4533,NaN,0.0,1.0,0.0,71.0,1.0,1.0,Fiber optic,0.0,1,1.0,0.0,1.0,1.0,Two year,1.0,Bank transfer (automatic),105.15,7555.00,0.0
3657,NaN,0.0,1.0,1.0,NaN,1.0,1.0,No,0.0,0,0.0,0.0,0.0,0.0,Two year,1.0,NaN,24.10,861.85,0.0
2651,NaN,0.0,0.0,NaN,24.0,1.0,0.0,Fiber optic,0.0,0,0.0,0.0,1.0,1.0,Month-to-month,1.0,NaN,90.35,2238.50,1.0
5811,NaN,0.0,1.0,0.0,61.0,1.0,1.0,DSL,0.0,1,1.0,0.0,1.0,1.0,Two year,0.0,NaN,80.90,4932.50,0.0
6947,NaN,0.0,NaN,1.0,68.0,1.0,1.0,Fiber optic,0.0,1,0.0,0.0,1.0,1.0,One year,1.0,Credit card (automatic),101.05,6770.50,0.0
1142,NaN,1.0,1.0,1.0,NaN,1.0,1.0,Fiber optic,0.0,0,0.0,0.0,0.0,0.0,Month-to-month,1.0,NaN,75.20,775.30,0.0
4728,NaN,0.0,0.0,0.0,46.0,0.0,0.0,DSL,1.0,0,1.0,1.0,0.0,0.0,Two year,0.0,NaN,38.25,1755.35,0.0
3978,NaN,0.0,0.0,0.0,3.0,1.0,0.0,NaN,0.0,0,0.0,0.0,1.0,1.0,Month-to-month,0.0,NaN,90.70,237.65,0.0
2116,Male,0.0,0.0,0.0,1.0,1.0,1.0,DSL,0.0,0,1.0,0.0,0.0,0.0,Month-to-month,0.0,Electronic check,54.35,54.35,1.0
6806,NaN,0.0,1.0,NaN,NaN,1.0,0.0,No,0.0,0,0.0,0.0,0.0,0.0,Two year,0.0,NaN,19.10,780.10,0.0


## Splitting Data

In [163]:
alca_df = alca_df.dropna(subset=["Churn"])

In [164]:
alca_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7041 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            3462 non-null   object 
 1   SeniorCitizen     7038 non-null   float64
 2   Partner           6690 non-null   float64
 3   Dependents        6682 non-null   float64
 4   tenure            6357 non-null   float64
 5   PhoneService      7039 non-null   float64
 6   MultipleLines     7037 non-null   float64
 7   InternetService   7035 non-null   object 
 8   OnlineSecurity    7039 non-null   float64
 9   OnlineBackup      7041 non-null   int64  
 10  DeviceProtection  7038 non-null   float64
 11  TechSupport       7040 non-null   float64
 12  StreamingTV       7040 non-null   float64
 13  StreamingMovies   7038 non-null   float64
 14  Contract          7041 non-null   object 
 15  PaperlessBilling  7039 non-null   float64
 16  PaymentMethod     3564 non-null   object 
 17  

In [165]:
X = alca_df.drop('Churn', axis=1)

y = alca_df['Churn']

In [166]:
X_train, X_test, y_train, y_test = train_test_split(X, y)

In [167]:
# First, let's pick out which columns are numeric vs. categorical
numeric_features = ["SeniorCitizen", "Partner", "Dependents", 'tenure', 'PhoneService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges']
categorical_features = ['MultipleLines', 'InternetService', 'Contract', 'PaymentMethod']

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),   # Fill missing with mean
    ("scaler", MinMaxScaler())                     # Scale to [0, 1]
])

categorical_transformer = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore")) 
])

# ColumnTransformer applies the correct transformer to each column set
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

###############################################################################
# 6. TRAIN A MODEL
# We'll use a Random Forest. We'll do a quick training with a pipeline.
###############################################################################

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [168]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
y_pred

Accuracy: 0.7910278250993753


array([1., 0., 0., ..., 0., 0., 0.])

In [169]:
tree_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=42))
])

tree_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [170]:
y_pred_tree = tree_model.predict(X_test)
accuracy_tree = accuracy_score(y_test, y_pred_tree)
print("Decision Tree Accuracy:", accuracy_tree)

Decision Tree Accuracy: 0.7382169222032936


In [171]:
forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

forest_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [172]:
y_pred_forest = forest_model.predict(X_test)
accuracy_forest = accuracy_score(y_test, y_pred_forest)
print("Accuracy:", accuracy)
print("Decision Tree Accuracy:", accuracy_tree)
print("Random Forest Accuracy:", accuracy_forest)

Accuracy: 0.7910278250993753
Decision Tree Accuracy: 0.7382169222032936
Random Forest Accuracy: 0.7910278250993753


In [173]:
b_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", GradientBoostingClassifier(n_estimators=100,random_state=42))
])

b_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [174]:
y_pred_b = b_model.predict(X_test)
accuracy_b = accuracy_score(y_test, y_pred_b)
print("Gradient Boosting Accuracy:", accuracy_b)
print("Decision Tree Accuracy:", accuracy_tree)
print("Random Forest Accuracy:", accuracy_forest)

Gradient Boosting Accuracy: 0.7802385008517888
Decision Tree Accuracy: 0.7382169222032936
Random Forest Accuracy: 0.7910278250993753


In [176]:
feature_names = forest_model.named_steps["preprocessor"].get_feature_names_out()


feat_imp_df = pd.Series(
    forest_model.named_steps["classifier"].feature_importances_,
    index=feature_names
)

print("\nFeature Importance:")
print(feat_imp_df)


Feature Importance:
num__SeniorCitizen                              0.021858
num__Partner                                    0.023174
num__Dependents                                 0.021274
num__tenure                                     0.150052
num__PhoneService                               0.006245
num__OnlineSecurity                             0.021443
num__OnlineBackup                               0.021080
num__DeviceProtection                           0.019283
num__TechSupport                                0.022187
num__StreamingTV                                0.018000
num__StreamingMovies                            0.017596
num__PaperlessBilling                           0.027111
num__MonthlyCharges                             0.174094
num__TotalCharges                               0.198016
cat__MultipleLines_0.0                          0.012071
cat__MultipleLines_1.0                          0.012208
cat__MultipleLines_nan                          0.000266
cat__Inter